# EventHallusion + VideoLLaVA on Colab

Notebook này chạy benchmark EventHallusion theo đúng mục tiêu Statement 4 và nhấn mạnh hơn vào tập **misleading**:

- lấy khoảng 200 video test, trong đó misleading chiếm nhiều nhất
- kiểm chứng `language prior bias` bằng cách giữ video cố định và đổi prompt
- kiểm chứng `context bias` bằng cách giữ prompt cố định và thử background removal
- so sánh `normal` và `spatial_gaussian` với nhiều mức `sigma` để thấy mức nhiễu mạnh/yếu

Output sẽ được lưu ra CSV/JSON trong thư mục kết quả để bạn đem đi phân tích.


In [ ]:
# Install dependencies
!pip -q install transformers accelerate bitsandbytes decord av huggingface_hub tqdm pandas opencv-python
!pip -q install git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d


In [ ]:
# Clone the benchmark repo
import os
repo_dir = "/content/EventHallusion"
if os.path.exists(repo_dir):
    !git -C /content/EventHallusion pull --rebase origin master
else:
    !git clone https://github.com/NguyenDucThang-tb/EventHallusion.git /content/EventHallusion
%cd /content/EventHallusion


In [ ]:
# Optional: mount Google Drive if you want to keep videos/results there
from google.colab import drive
drive.mount('/content/drive')


## Download videos

Chạy cell dưới đây nếu bạn chưa có thư mục video local. `questions/` chỉ là JSON câu hỏi, còn video phải tải riêng từ Google Drive của EventHallusion.

Sau khi giải nén, notebook sẽ tự tìm toàn bộ file `.mp4` bên trong thư mục `/content/EventHallusion`.

In [ ]:
# Download and extract EventHallusion videos
!pip -q install gdown
!gdown --fuzzy "https://drive.google.com/file/d/1IPmx6Y80UrXwVPmZJh6zjCPHtlsw4p9n/view?usp=sharing" -O EventHallusion_videos.zip
!mkdir -p /content/EventHallusion/videos
!unzip -q EventHallusion_videos.zip -d /content/EventHallusion/videos


## Prepare data

You need the EventHallusion videos extracted to a folder. The files in `questions/` are only annotations / question sets, not videos.

The video folder is not included in the repo checkout. Download the EventHallusion video release from the Google Drive link in the README, then extract it to `/content/EventHallusion/videos` or to a Drive path you mount in Colab.

Typical paths:

- questions: `/content/EventHallusion/questions`
- videos: `/content/EventHallusion/videos`  # create this by extracting the downloaded video archive
- results: `/content/EventHallusion/results`

If your videos are on Drive, set `video_root` to that path instead.

In [ ]:
# Load Video-LLaVA
import torch
from transformers import VideoLlavaForConditionalGeneration, VideoLlavaProcessor, BitsAndBytesConfig

model_name = "LanguageBind/Video-LLaVA-7B-hf"
processor = VideoLlavaProcessor.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = VideoLlavaForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded")


In [ ]:
# Reload the helper and validate the data paths before running the benchmark
from pathlib import Path
import importlib
import eventhallusion_videollava_eval

importlib.reload(eventhallusion_videollava_eval)
from eventhallusion_videollava_eval import compare_conditions

questions_root = Path('/content/EventHallusion/questions')
video_root = Path('/content/EventHallusion/videos')
print('questions exists:', questions_root.exists())
print('videos exists:', video_root.exists())
print('question files:', sorted([p.name for p in questions_root.glob('*.json')]))
videos = sorted(video_root.rglob('*.mp4'))
print('video count:', len(videos))
print('sample videos:', [p.name for p in videos[:5]])


In [ ]:
# Run EventHallusion benchmark
questions_root = "/content/EventHallusion/questions"
video_root = "/content/EventHallusion/videos"
out_dir = "/content/EventHallusion/results"

# Main benchmark: focus more on misleading videos
summary = compare_conditions(
    model=model,
    processor=processor,
    questions_root=questions_root,
    video_root=video_root,
    out_dir=out_dir,
    n_total_videos=200,
    n_frames=4,
    sigma=25,
    seed=42,
    per_split={"misleading": 100, "entire": 50, "mix": 50},
    show_progress=True,
)

summary


## Spatial Strength Check

`spatial_gaussian` là additive Gaussian noise theo từng pixel: `N(0, sigma^2)`.
Nếu `sigma` nhỏ so với dải pixel 8-bit (`0..255`), video sau nhiễu vẫn khá giống video gốc nên kết quả có thể ít thay đổi.


In [ ]:
from eventhallusion_videollava_eval import summarize_spatial_gaussian

for sigma in [5, 10, 25, 50, 75]:
    print(summarize_spatial_gaussian(sigma))


## Language Bias

Giữ video cố định, đổi prompt bằng cách thêm context prefix như `At the beach,` hoặc `At school,`.
Ta đo thêm `Language Consistency Score` = tỉ lệ prediction giống nhau giữa prompt gốc và prompt có context mới.


In [ ]:
from eventhallusion_videollava_eval import run_language_bias_eval, save_results

language_df, language_metrics = run_language_bias_eval(
    model=model,
    processor=processor,
    questions_root=questions_root,
    video_root=video_root,
    n_total_videos=100,
    n_frames=4,
    seed=42,
    split="misleading",
    context_prefixes=[
        "At the beach,",
        "At school,",
        "In a kitchen,",
        "In an office,",
    ],
)

save_results(language_df, out_dir, "eventhallusion_language_bias")
language_metrics


## Context Bias

Giữ prompt cố định nhưng thử background removal (foreground mask) thay vì video gốc.
Ta đo thêm tỉ lệ prediction giống nhau giữa hai điều kiện.


In [ ]:
from eventhallusion_videollava_eval import run_context_bias_eval

context_df, context_metrics = run_context_bias_eval(
    model=model,
    processor=processor,
    questions_root=questions_root,
    video_root=video_root,
    n_total_videos=100,
    n_frames=4,
    seed=42,
    split="misleading",
    use_background_removal=True,
)

save_results(context_df, out_dir, "eventhallusion_context_bias")
context_metrics


## What to look at

- `acc_misleading`: language prior / context bias
- `acc_entire`: whether the model follows the whole video timeline
- `acc_mix`: whether the model can avoid frame-level shortcut on mixed rare/common events

Compare `normal` vs `spatial_gaussian`.
If spatial Gaussian helps mostly on `misleading`, then it mainly attacks spatial/context shortcuts rather than temporal reasoning itself.

In [ ]:
# Download results to Drive if needed
import shutil
drive_out = "/content/drive/MyDrive/EventHallusion_results"
os.makedirs(drive_out, exist_ok=True)
if os.path.exists(out_dir):
    for name in [
        "eventhallusion_summary.csv",
        "eventhallusion_normal.csv",
        "eventhallusion_spatial_gaussian.csv",
        "eventhallusion_normal_predictions.json",
        "eventhallusion_spatial_gaussian_predictions.json",
        "eventhallusion_language_bias.csv",
        "eventhallusion_language_bias.json",
        "eventhallusion_context_bias.csv",
        "eventhallusion_context_bias.json",
    ]:
        src = os.path.join(out_dir, name)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(drive_out, name))
print(drive_out)
